In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsRegressor as KNR
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score
import itertools
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR



In [2]:
df = pd.read_csv("fixed_final_data_product.csv")

In [50]:
print(len(df))

1032


# Clean Dataset

In [3]:
target_col = "Target_CURRENTLAPTIMEINMS"
prefixes = ['BPS', 'STS', 'BPE', 'STM', 'STE', 'THE', 'THS']

driver_input = ['BPS_SPEED', 'BPS_THROTTLE', 'BPS_STEER', 'BPS_BRAKE', 'BPS_CURRENTLAPTIMEINMS', 
             'BPS_LAPDISTANCE', 'BPS_WORLDPOSITIONX', 'BPS_WORLDPOSITIONY', 'BPE_SPEED', 'BPE_THROTTLE', 
             'BPE_STEER', 'BPE_BRAKE', 'BPE_CURRENTLAPTIMEINMS', 'BPE_LAPDISTANCE', 'BPE_WORLDPOSITIONX', 'BPE_WORLDPOSITIONY',
            'THS_SPEED', 'THS_THROTTLE', 'THS_STEER', 'THS_BRAKE', 'THS_CURRENTLAPTIMEINMS', 'THS_LAPDISTANCE', 'THS_WORLDPOSITIONX', 
             'THS_WORLDPOSITIONY', 'THE_SPEED', 'THE_THROTTLE', 'THE_STEER', 'THE_BRAKE', 'THE_CURRENTLAPTIMEINMS', 
             'THE_LAPDISTANCE', 'THE_WORLDPOSITIONX', 'THE_WORLDPOSITIONY', 'STS_SPEED', 'STS_THROTTLE', 'STS_STEER', 'STS_BRAKE', 
             'STS_CURRENTLAPTIMEINMS', 'STS_LAPDISTANCE', 'STS_WORLDPOSITIONX', 'STS_WORLDPOSITIONY', 'STM_SPEED', 'STM_THROTTLE', 
             'STM_STEER', 'STM_BRAKE', 'STM_CURRENTLAPTIMEINMS', 'STM_LAPDISTANCE', 'STM_WORLDPOSITIONX', 'STM_WORLDPOSITIONY', 
            'STE_SPEED', 'STE_THROTTLE', 'STE_STEER', 'STE_BRAKE', 'STE_CURRENTLAPTIMEINMS', 'STE_LAPDISTANCE', 'STE_WORLDPOSITIONX', 
             'STE_WORLDPOSITIONY', 'Target_CURRENTLAPTIMEINMS']

# Keep only driver related features
df = df[[col for col in driver_input if col in df.columns]]

In [4]:
def apply_iqr_filter(df, prefixes_to_clean):
    df_filtered = df.copy()
    
    for prefix in prefixes_to_clean:
        lap_col = f"{prefix}_CURRENTLAPTIMEINMS"
        
        if lap_col in df_filtered.columns:
            Q1 = df_filtered[lap_col].quantile(0.25)
            Q3 = df_filtered[lap_col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            df_filtered = df_filtered[(df_filtered[lap_col] >= lower_bound) & (df_filtered[lap_col] <= upper_bound)]
    
    return df_filtered


In [5]:
# Removing laps whose times are statistically too high
df = apply_iqr_filter(df, ['Target'])

In [6]:
# Removing rows will Null values in any column
df = df.dropna()


## Finding Subset of Features that can be cleaned

In [7]:
def evaluate_model(y_true, y_pred, X_test):
    n = X_test.shape[0]
    p = X_test.shape[1]
    mse = mean_squared_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    r2_adj = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    
    return mse, rmse, mae, r2, r2_adj

In [8]:
X = df.drop(columns=target_col)
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
results_all = {}
models = {
    "KNNR": KNR(n_neighbors=5),
    "SVR": SVR(kernel='rbf')
}

for model_name, model in models.items():
    results = []
    for r in range(len(prefixes) + 1):
        for combo in itertools.combinations(prefixes, r):
            # Apply IQR filtering based on training data
            train_filtered = apply_iqr_filter(X_train, combo)
            test_filtered = apply_iqr_filter(X_test, combo)

            # Skip if filtering removes everything
            if train_filtered.empty or test_filtered.empty:
                continue

            # Match y indices
            y_train_filtered = y_train.loc[train_filtered.index]
            y_test_filtered = y_test.loc[test_filtered.index]

            # Scale features
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(train_filtered)
            X_test_scaled = scaler.transform(test_filtered)

            # Fit model and predict
            model.fit(X_train_scaled, y_train_filtered)
            y_pred = model.predict(X_test_scaled)
            
            # Evaluate metrics
            mse, rmse, mae, r2, r2_adj = evaluate_model(y_test_filtered, y_pred, test_filtered)
            results.append((combo, mse, rmse, mae, r2, r2_adj))
    
    # Store results dataframe
    results_df = pd.DataFrame(results, columns=["prefixes", "MSE", "RMSE", "MAE", "R2", "R2_adj"])
    best_row = results_df.loc[results_df["MSE"].idxmin()]
    results_all[model_name] = best_row


for model_name, best_row in results_all.items():
    print(f"\n=== Best Result for {model_name} ===")
    print(f"Best prefixes to clean: {best_row['prefixes'] or 'no prefixes'}")
    print(f"Lowest MSE: {best_row['MSE']:.2f}")
    print(f"RMSE: {best_row['RMSE']:.2f}")
    print(f"MAE: {best_row['MAE']:.2f}")
    print(f"R²: {best_row['R2']:.4f}")
    print(f"Adjusted R²: {best_row['R2_adj']:.4f}")



=== Best Result for KNNR ===
Best prefixes to clean: ('STM', 'STE', 'THS')
Lowest MSE: 204541.83
RMSE: 452.26
MAE: 297.88
R²: 0.5695
Adjusted R²: 0.1060

=== Best Result for SVR ===
Best prefixes to clean: ('BPS', 'STM', 'STE', 'THS')
Lowest MSE: 433294.03
RMSE: 658.25
MAE: 484.55
R²: 0.0074
Adjusted R²: -1.2852


In [10]:
print(y_train_filtered)

585    12711
181    12936
6      12729
912    12696
197    12488
       ...  
858    12721
22     12431
145    12976
363    12250
141    13231
Name: Target_CURRENTLAPTIMEINMS, Length: 360, dtype: int64


In [11]:
# Applying filtering to the prefixes for KNN
KNNR_df = apply_iqr_filter(df, ["STM", "STE", "THS"])
SVR_df = apply_iqr_filter(df, ['BPS', 'STM', 'STE', 'THS'])

# Modelling

In [12]:
def print_model_outcomes(mse, rmse, mae, r2, r2_adj):
    print(f"Lowest MSE: {mse:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R²: {r2:.4f}")
    print(f"Adjusted R²: {r2_adj:.4f}")

## KNN Regressor

In [13]:
X = KNNR_df.drop(columns=target_col).copy()
y = KNNR_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Hyperparameter Not Tuned

In [14]:
# 5 neighbours are commonly used
model = KNR(n_neighbors=5)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

mse, rmse, mae, r2, r2_adj = evaluate_model(y_test, y_pred, X_test)
print_model_outcomes(mse, rmse, mae, r2, r2_adj)

Lowest MSE: 220519.89
RMSE: 469.60
MAE: 314.70
R²: 0.5254
Adjusted R²: 0.0333


### Hyperparameter Tuning

In [15]:
param_grid = {
    'n_neighbors': range(1, 58),
    'weights': ['uniform', 'distance']
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    KNR(),
    param_grid,
    cv=kf,
    scoring='neg_mean_squared_error',
)

grid_search.fit(X_train_scaled, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV MSE:", -grid_search.best_score_)


Best parameters: {'n_neighbors': 6, 'weights': 'distance'}
Best CV MSE: 233125.11404336692


In [16]:
final_model = KNR(n_neighbors=6, weights='distance')
final_model.fit(X_train_scaled, y_train)

y_pred = final_model.predict(X_test_scaled)

mse, rmse, mae, r2, r2_adj = evaluate_model(y_test, y_pred, X_test)
print_model_outcomes(mse, rmse, mae, r2, r2_adj)

Lowest MSE: 219623.44
RMSE: 468.64
MAE: 312.08
R²: 0.5274
Adjusted R²: 0.0372


# SVM Regression

In [17]:
X = SVR_df.drop(columns=target_col).copy()
y = SVR_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Hyperparameter Not Tuned

In [18]:

svr = SVR(kernel='rbf')
svr.fit(X_train_scaled, y_train)
y_pred = svr.predict(X_test_scaled)

mse, rmse, mae, r2, r2_adj = evaluate_model(y_test, y_pred, X_test)
print_model_outcomes(mse, rmse, mae, r2, r2_adj)

Lowest MSE: 300882.02
RMSE: 548.53
MAE: 411.81
R²: -0.0373
Adjusted R²: -1.3574


### Hyperparameter Tuned

In [19]:

svr = SVR()
param_grid = [
    {
        'kernel': ['rbf'], 
        'C': [1, 10, 100, 500, 1000, 1500, 2000],
        'gamma': [0.001, 0.01, 0.1, 1, 10],
        'epsilon': [10, 50, 100, 200, 500, 1000]
    },
    {
        'kernel': ['linear'],
        'C': [1, 10, 100, 500, 1000, 1500, 2000, 5000],
        'epsilon': [10, 50, 100, 200, 500, 1000]
    },
    {
        'kernel': ['poly'],
        'C': [1, 10, 100, 500, 1000, 1500, 2000],
        'gamma': [0.001, 0.01, 0.1, 1, 10],
        'degree': [2, 3, 4],
        'epsilon': [10, 50, 100, 200, 500, 1000]
    }
]


grid_search = GridSearchCV(svr, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best score:", -grid_search.best_score_)



Best parameters: {'C': 1000, 'epsilon': 100, 'kernel': 'linear'}
Best score: 34936.205396327525


In [20]:
svr = SVR(kernel='linear', C = 1000, epsilon= 100)
svr.fit(X_train_scaled, y_train)
y_pred = svr.predict(X_test_scaled)

mse, rmse, mae, r2, r2_adj = evaluate_model(y_test, y_pred, X_test)
print_model_outcomes(mse, rmse, mae, r2, r2_adj)

Lowest MSE: 9718.65
RMSE: 98.58
MAE: 75.68
R²: 0.9665
Adjusted R²: 0.9239


In [27]:
feature_names = X_train.columns  # from your original DataFrame
feature_importances = abs(svr.coef_[0])
importance_series = pd.Series(feature_importances, index=feature_names).sort_values(ascending=False)

print("Feature importances:\n", importance_series)


Feature importances:
 STE_CURRENTLAPTIMEINMS    688.044157
STE_SPEED                 522.073634
STE_LAPDISTANCE           414.052976
STS_WORLDPOSITIONY        147.995286
THS_CURRENTLAPTIMEINMS    127.883486
STS_WORLDPOSITIONX        125.366922
THS_WORLDPOSITIONY        117.009200
BPS_LAPDISTANCE           103.989728
BPE_SPEED                 102.571636
BPS_CURRENTLAPTIMEINMS     84.604866
BPE_CURRENTLAPTIMEINMS     77.879524
STS_LAPDISTANCE            73.561879
STS_SPEED                  62.681767
STE_WORLDPOSITIONY         57.411564
THE_LAPDISTANCE            57.190860
THE_CURRENTLAPTIMEINMS     56.320399
BPE_WORLDPOSITIONX         53.358776
STM_WORLDPOSITIONX         51.587392
STE_WORLDPOSITIONX         48.715794
THS_WORLDPOSITIONX         43.496673
THS_SPEED                  40.381190
BPE_WORLDPOSITIONY         40.151547
STM_CURRENTLAPTIMEINMS     35.461146
BPS_WORLDPOSITIONX         35.354105
THS_LAPDISTANCE            33.189217
STE_THROTTLE               32.602918
THE_BRAKE       